# 📄 Dispatcher Service — Interactive Notebook

This notebook lets you run and test each step of the Dispatcher pipeline interactively.

**Pipeline Steps:**
1. Install dependencies
2. Configure credentials
3. Extract text from a PDF (local file or GCS)
4. Classify the document using Gemini Flash
5. Simulate Pub/Sub routing

## Step 1 — Install Dependencies

In [ ]:
!pip install google-cloud-pubsub google-cloud-storage google-generativeai pypdf pydantic --quiet

## Step 2 — Configuration

Set your credentials here. Get a free Gemini API key from https://aistudio.google.com/apikey

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
PROJECT_ID     = "ml-deployment-482112-490513"   # Your GCP project ID
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"       # From aistudio.google.com/apikey
GCS_BUCKET     = f"raw-documents-{PROJECT_ID}-dev"  # Your upload bucket

# Pub/Sub topic routing
INVOICE_TOPIC     = "process-invoice"
CONTRACT_TOPIC    = "process-contract"
DEAD_LETTER_TOPIC = "dead-letter-topic"

print(f"Project : {PROJECT_ID}")
print(f"Bucket  : {GCS_BUCKET}")
print(f"API Key : {'SET ✅' if GEMINI_API_KEY != 'YOUR_GEMINI_API_KEY_HERE' else 'NOT SET ❌'}")

## Step 3 — Initialize Clients

In [ ]:
import io, json
from google.cloud import pubsub_v1, storage
import google.generativeai as genai
from pypdf import PdfReader

# GCP clients
storage_client = storage.Client(project=PROJECT_ID)
publisher      = pubsub_v1.PublisherClient()

# Gemini client
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")

print("All clients initialized ✅")

## Step 4a — Extract text from a LOCAL PDF file

Use this cell to test with a PDF on your machine.

In [ ]:
LOCAL_PDF_PATH = r"C:\path\to\your\document.pdf"   # <-- Change this

with open(LOCAL_PDF_PATH, "rb") as f:
    reader = PdfReader(f)
    doc_text = "\n".join(page.extract_text() or "" for page in reader.pages)

doc_text = doc_text[:2000]  # Trim to stay within token limits

print(f"Extracted {len(doc_text)} characters")
print("─" * 60)
print(doc_text[:500], "...")  # Preview first 500 chars

## Step 4b — (Alternative) Extract text from GCS

Use this if the PDF is already uploaded to your Cloud Storage bucket.

In [ ]:
GCS_FILENAME = "test-invoice.pdf"   # <-- Change to your filename in GCS

blob      = storage_client.bucket(GCS_BUCKET).blob(GCS_FILENAME)
pdf_bytes = blob.download_as_bytes()
reader    = PdfReader(io.BytesIO(pdf_bytes))
doc_text  = "\n".join(page.extract_text() or "" for page in reader.pages)
doc_text  = doc_text[:2000]

print(f"Downloaded gs://{GCS_BUCKET}/{GCS_FILENAME}")
print(f"Extracted {len(doc_text)} characters")
print("─" * 60)
print(doc_text[:500], "...")

## Step 5 — Classify with Gemini Flash

In [ ]:
CLASSIFICATION_PROMPT = """You are a document classifier. Read the document text and identify its type.
Respond with ONLY one of these exact words: INVOICE, CONTRACT, TAX_FORM, UNKNOWN.
No explanation. No punctuation. Just the single word."""

response = model.generate_content(
    f"{CLASSIFICATION_PROMPT}\n\nDocument text:\n\n{doc_text}"
)
doc_type = response.text.strip().upper()

print(f"🤖 Gemini classified this document as: {doc_type}")

## Step 6 — Simulate Pub/Sub Routing

This shows which topic the document would be routed to. Set `DRY_RUN = False` to actually publish.

In [ ]:
DRY_RUN  = True   # Set to False to actually publish to Pub/Sub
FILENAME = GCS_FILENAME if 'GCS_FILENAME' in dir() else LOCAL_PDF_PATH

# Determine destination topic
if "INVOICE" in doc_type:
    target_topic = INVOICE_TOPIC
elif "CONTRACT" in doc_type:
    target_topic = CONTRACT_TOPIC
else:
    target_topic = DEAD_LETTER_TOPIC

payload = {
    "gcs_uri":       f"gs://{GCS_BUCKET}/{FILENAME}",
    "bucket":        GCS_BUCKET,
    "filename":      FILENAME,
    "document_type": doc_type,
}

print(f"📨 Would route to topic : {target_topic}")
print(f"📦 Payload              : {json.dumps(payload, indent=2)}")

if not DRY_RUN:
    topic_path = publisher.topic_path(PROJECT_ID, target_topic)
    future     = publisher.publish(topic_path, json.dumps(payload).encode("utf-8"))
    msg_id     = future.result()
    print(f"\n✅ Published! Message ID: {msg_id}")
else:
    print("\n⚠️  DRY_RUN=True — set to False to actually publish.")

## Step 7 — Simulate a Full Pub/Sub Push Message

This simulates the exact JSON payload Pub/Sub sends to the Cloud Run dispatcher endpoint.

In [ ]:
import base64, requests

DISPATCHER_URL = "https://dispatcher-699339551124.us-central1.run.app/"
TEST_FILENAME  = "test-invoice.pdf"   # Must exist in your GCS bucket

# Build the simulated GCS event payload
gcs_event = json.dumps({"bucket": GCS_BUCKET, "name": TEST_FILENAME})
encoded   = base64.b64encode(gcs_event.encode()).decode()

pubsub_message = {"message": {"data": encoded}}

print("Sending simulated Pub/Sub event to dispatcher...")
resp = requests.post(DISPATCHER_URL, json=pubsub_message)

print(f"Status : {resp.status_code}")
print(f"Response: {resp.json()}")